# ML-08: Capstone Modeling, ranking pages by decline risk

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 2, Refresh scoring. The job is to rank visible pages by how likely they are to be losing traffic, so an editor can work the top of the list first. This notebook is the distilled result of a wider search: I built a greedy experiment harness under `work/experiments/` that tried roughly fifty methods one at a time across preprocessing, feature engineering, sampling, and twelve model families, keeping a change only when it beat the current best on paired client-grouped folds. Here I keep the reasoning and the one change that survived re-testing on fresh folds and a held-out set of clients.


## 1. Problem and label

A page is declining when its measured traffic trend points down. I take that directly from the warehouse: `is_declining = 1` when `trend_direction == "down"`. The population is pages with real exposure, `impressions_90d >= 100`, because ranking a page nobody sees is not a refresh decision anyone acts on.

The metric that matters for the workflow is Precision@50: of the fifty pages the model flags first, how many are truly declining. ROC-AUC rides alongside as the overall ranking quality. Base rate is high, near 0.60, so a model has to clear that floor to be worth anything.

In [1]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
visible = df[df["impressions_90d"] >= 100]
print("all pages:", len(df), " visible:", len(visible), " clients:", df["client_id"].nunique())
print("visible base rate:", round(visible["is_declining"].mean(), 3))
print("sub-threshold base rate:", round(df[df["impressions_90d"] < 100]["is_declining"].mean(), 3))

all pages: 30000  visible: 22006  clients: 32
visible base rate: 0.598
sub-threshold base rate: 0.389


## 2. Leakage policy

The label is built from the recent traffic window, so anything measured over that same window would let the model read the answer off its own inputs. I exclude every 90-day and last-30-day engagement column, the ratios derived from them, and the two trend columns the label comes from. What the model is allowed to see is static page properties and the prior 30-day window, which sits before the label window. `impressions_90d` earns one narrow exception: it defines who is in the population and which rows are scored, and it never enters the model as a feature.

| Kept out of the model | Why |
|---|---|
| `trend_direction`, `trend_pct` | the label itself |
| `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d` | label-window traffic |
| `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d` | label-window engagement |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | overlaps the label window |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | label-window ratios |
| `impression_tier`, `position_tier`, `days_with_impressions`, `days_with_sessions` | derived from label-window traffic |

Allowed: content age and freshness, word and character counts, keyword economics (`search_volume`, `competition`, `cpc`), page type and intent, and the prior-window counts `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`.

In [2]:
num = ["content_age_days", "days_since_last_update", "word_count", "char_count",
       "search_volume", "competition", "cpc",
       "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
cats = ["content_type", "main_intent", "competition_level"]

def features(frame):
    f = frame.copy()
    for c in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
        f["log_" + c] = np.log1p(f[c].fillna(0))
    f["has_keyword"] = f["search_volume"].notna().astype(float)
    f["has_word_count"] = f["word_count"].notna().astype(float)
    numcols = num + ["log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d",
                     "has_keyword", "has_word_count"]
    X = pd.concat([f[numcols], f[cats].astype("object").fillna("unknown")], axis=1)
    return X, numcols, cats

def model():
    pre = ColumnTransformer([
        ("n", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value=0)),
                        ("sc", StandardScaler())]),
         num + ["log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d",
                "has_keyword", "has_word_count"]),
        ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats)])
    return Pipeline([("pre", pre),
                     ("lr", LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0,
                                               random_state=42))])

## 3. Split design and a held-out lockbox

A page carries its client's fingerprint: shared templates, one niche, the same tracking setup. A random split lets a model memorise the client and report skill it does not have, so the honest question is whether it ranks pages for a client it has never seen. Every split here keeps whole clients together.

For the search itself I use fifteen client-grouped folds, three seeded partitions of five, and score every candidate on the identical folds so comparisons are paired. Because the harness reuses those folds for dozens of decisions, the fold numbers drift optimistic over time. To keep one unbiased read, I set aside six clients as a lockbox before any search ran, chosen once by a fixed seed for size and base rate and never looked at until the very end. The lockbox is scored exactly twice: the starting model and the final one.

In [3]:
clients = np.array(sorted(df["client_id"].unique()))
rng = np.random.default_rng(20260715)
lockbox = None
for _ in range(100000):
    pick = set(rng.choice(clients, 6, replace=False).tolist())
    sub = visible[visible["client_id"].isin(pick)]
    if 3000 <= len(sub) <= 5500 and 0.55 <= sub["is_declining"].mean() <= 0.65:
        lockbox = pick
        break
dev = set(clients) - lockbox
print("lockbox clients:", len(lockbox), " dev clients:", len(dev))
print("lockbox visible rows:", len(visible[visible["client_id"].isin(lockbox)]),
      " base rate:", round(visible[visible["client_id"].isin(lockbox)]["is_declining"].mean(), 3))

def grouped_folds(groups, seed, n=5):
    gs = pd.Series(groups)
    order = np.random.default_rng(seed).permutation(np.array(sorted(gs.unique())))
    sizes = gs.value_counts()
    load = np.zeros(n)
    fold_of = {}
    for c in order:
        fi = int(np.argmin(load))
        fold_of[c] = fi
        load[fi] += sizes[c]
    fid = gs.map(fold_of).values
    return [(np.where(fid != f)[0], np.where(fid == f)[0]) for f in range(n)]

lockbox clients: 6  dev clients: 26
lockbox visible rows: 5284  base rate: 0.605


## 4. Method choice and the one change that held

I want a good ranker, not a classifier tuned for raw accuracy, and I want something an editor can read. Logistic Regression is the model I keep. The harness put it head to head with Random Forest, Extra Trees, Gradient Boosting, HistGradientBoosting, k-nearest-neighbours, LDA, Naive Bayes and a small MLP; none of them ranked the top fifty pages better, and several held their AUC only by scrambling that top fifty, which is the part the workflow actually uses.

The change that survived is about data, not the model. The visible population is 22,006 pages, but the warehouse holds another 7,994 below the visibility threshold, all still labelled. Training on those extra pages as well, while still scoring only the visible ones, gives the model more of the same leakage-safe signal to learn from. Everything else I tried, engineered ratios and interactions, per-client rank features, alternative scalers and imputers, sampling and class-weight schemes, either did nothing or looked good on the search folds and then evaporated on fresh folds. Widening did not evaporate.

In [4]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

dev_all = df[df["client_id"].isin(dev)].reset_index(drop=True)
dev_vis = dev_all[dev_all["impressions_90d"] >= 100].reset_index(drop=True)

def run(train_pop):
    rows_auc, rows_p50 = [], []
    for seed in (0, 1, 2):
        for tr_c, te_c in grouped_folds(dev_all["client_id"].values, seed):
            train_clients = set(dev_all["client_id"].values[tr_c])
            test_clients = set(dev_all["client_id"].values[te_c])
            base = dev_all if train_pop == "widened" else dev_vis
            tr = base[base["client_id"].isin(train_clients)]
            te = dev_vis[dev_vis["client_id"].isin(test_clients)]
            Xtr, _, _ = features(tr)
            Xte, _, _ = features(te)
            m = model()
            m.fit(Xtr, tr["is_declining"].values)
            s = m.predict_proba(Xte)[:, 1]
            rows_auc.append(roc_auc_score(te["is_declining"].values, s))
            rows_p50.append(precision_at_k(s, te["is_declining"].values))
    return np.array(rows_auc), np.array(rows_p50)

base_auc, base_p50 = run("visible")
wide_auc, wide_p50 = run("widened")

def staleness_scores():
    a, p = [], []
    for seed in (0, 1, 2):
        for _, te_c in grouped_folds(dev_all["client_id"].values, seed):
            test_clients = set(dev_all["client_id"].values[te_c])
            te = dev_vis[dev_vis["client_id"].isin(test_clients)]
            s = te["days_since_last_update"].values
            a.append(roc_auc_score(te["is_declining"].values, s))
            p.append(precision_at_k(s, te["is_declining"].values))
    return np.array(a), np.array(p)

stale_auc, stale_p50 = staleness_scores()

table = pd.DataFrame([
    ["staleness floor", stale_auc.mean(), stale_p50.mean()],
    ["LogReg, visible-train (baseline)", base_auc.mean(), base_p50.mean()],
    ["LogReg, widened-train (final)", wide_auc.mean(), wide_p50.mean()],
], columns=["method", "ROC_AUC", "P@50"]).round(3)
paired = wide_auc - base_auc
print(table.to_string(index=False))
print("\nwidened vs baseline, paired over 15 folds: mean dAUC",
      round(paired.mean(), 4), " wins", int((paired > 0).sum()), "of 15")

                          method  ROC_AUC  P@50
                 staleness floor    0.495 0.712
LogReg, visible-train (baseline)    0.603 0.824
   LogReg, widened-train (final)    0.624 0.855

widened vs baseline, paired over 15 folds: mean dAUC 0.021  wins 9 of 15


The staleness floor confirms the task is hard: how long since a page was last touched barely separates decline from the rest. Against the visible-train baseline, widening the training population lifts Precision@50 by a few points and nudges AUC up, winning a clear majority of the paired folds. The gain is modest in AUC because the leakage-safe ceiling here is genuinely low, but it lands where the workflow reads: the top of the list.

## 5. The lockbox, read once

Everything above ran on the development clients. These six clients were held out from the entire search. I train each model on all development clients and score the lockbox a single time.

In [5]:
def lockbox_eval(train_pop):
    base = dev_all if train_pop == "widened" else dev_vis
    lb = visible[visible["client_id"].isin(lockbox)]
    Xtr, _, _ = features(base)
    Xte, _, _ = features(lb)
    m = model()
    m.fit(Xtr, base["is_declining"].values)
    s = m.predict_proba(Xte)[:, 1]
    return roc_auc_score(lb["is_declining"].values, s), precision_at_k(s, lb["is_declining"].values)

lb_base = lockbox_eval("visible")
lb_wide = lockbox_eval("widened")
print(pd.DataFrame([
    ["LogReg, visible-train", round(lb_base[0], 3), round(lb_base[1], 3)],
    ["LogReg, widened-train", round(lb_wide[0], 3), round(lb_wide[1], 3)],
], columns=["method", "lockbox ROC_AUC", "lockbox P@50"]).to_string(index=False))

               method  lockbox ROC_AUC  lockbox P@50
LogReg, visible-train            0.666          0.76
LogReg, widened-train            0.668          0.84


On clients the search never touched, widened training holds its AUC and moves Precision@50 up by a clear margin. That is the headline: the change generalises to new clients on the metric the refresh queue is built from, and it does not owe its gain to overfitting the development folds.

## 6. Where the model is right and wrong

A single AUC hides how uneven this is across clients. I score each held-out client on its own out-of-fold predictions from the final model, then look at what the top of the queue is made of.

In [6]:
oof = np.full(len(dev_vis), np.nan)
for tr_c, te_c in grouped_folds(dev_all["client_id"].values, 0):
    train_clients = set(dev_all["client_id"].values[tr_c])
    test_clients = set(dev_all["client_id"].values[te_c])
    tr = dev_all[dev_all["client_id"].isin(train_clients)]
    te_mask = dev_vis["client_id"].isin(test_clients).values
    Xtr, _, _ = features(tr)
    Xte, _, _ = features(dev_vis[te_mask])
    m = model()
    m.fit(Xtr, tr["is_declining"].values)
    oof[te_mask] = m.predict_proba(Xte)[:, 1]

per_client = []
for c, g in dev_vis.groupby("client_id"):
    idx = dev_vis["client_id"].eq(c).values
    y = dev_vis.loc[idx, "is_declining"].values
    if len(np.unique(y)) == 2:
        per_client.append((c, len(y), roc_auc_score(y, oof[idx])))
pc = pd.DataFrame(per_client, columns=["client", "pages", "auc"]).sort_values("auc")
print("per-client held-out AUC: min", round(pc["auc"].min(), 2),
      " median", round(pc["auc"].median(), 2), " max", round(pc["auc"].max(), 2))
print("clients ranked worse than chance:", int((pc["auc"] < 0.5).sum()), "of", len(pc))

per-client held-out AUC: min 0.44  median 0.63  max 0.82
clients ranked worse than chance: 1 of 22


In [7]:
top = dev_vis.iloc[np.argsort(-oof)[:50]]
print("top-50 flagged pages, share truly declining:", round(top["is_declining"].mean(), 2))
print("\nby content type:")
print(top.groupby("content_type")["is_declining"].agg(["size", "mean"]).round(2).to_string())
print("\nby age tier:")
print(top.groupby("age_tier")["is_declining"].agg(["size", "mean"]).round(2).to_string())

top-50 flagged pages, share truly declining: 0.78

by content type:
                 size  mean
content_type               
keyword article    50  0.78

by age tier:
          size  mean
age_tier            
181-365     19  0.74
91-180      31  0.81


In [8]:
final = model()
Xall, _, _ = features(dev_all)
final.fit(Xall, dev_all["is_declining"].values)
names = final.named_steps["pre"].get_feature_names_out()
coef = pd.Series(final.named_steps["lr"].coef_[0], index=names).sort_values()
print("pushes toward decline:")
print(coef.tail(6).round(3).to_string())
print("\npushes away from decline:")
print(coef.head(6).round(3).to_string())

pushes toward decline:
n__clicks_prev_30d              0.129
c__main_intent_unknown          0.154
n__has_word_count               0.322
n__has_keyword                  0.366
c__competition_level_unknown    0.654
n__log_impressions_prev_30d     1.211

pushes away from decline:
n__log_clicks_prev_30d        -0.720
c__competition_level_LOW      -0.386
n__content_age_days           -0.341
c__main_intent_navigational   -0.293
c__competition_level_MEDIUM   -0.172
n__char_count                 -0.143


Per-client AUC swings from near-random to strong: the model reads some clients well and is close to a coin flip on others, and the pooled number averages over that. Content type cannot slice the queue here because the visible corpus is almost entirely keyword articles, so the age-tier cut is the informative one, and the top fifty lean toward pages in their first year rather than the oldest bracket. The coefficients read the way the domain would expect: pages that pulled real prior-window impressions have the most traffic to lose and score toward decline, while those still converting that traffic into clicks score away from it.

## 7. Limitations, and what a higher number would cost

The ceiling for this label, with every leakage-safe rule enforced, is around 0.60 AUC. I confirmed that the hard way: a wide automated search over preprocessing, features, sampling and a dozen model families moved the pooled AUC by thousandths, and most of what looked like progress on the search folds did not survive fresh folds or the lockbox.

It is easy to make the AUC look much better, and worth showing exactly how, because the how is the whole point. If I let the model see the label-window columns it is supposed to be blind to, the same Logistic Regression jumps from the low 0.60s to above 0.90.

In [9]:
leaked = ["ctr", "avg_position", "engagement_rate", "impressions_last_30d",
          "clicks_last_30d", "sessions_last_30d", "impressions_90d", "clicks_90d"]

def auc_with(extra):
    a, p = [], []
    for seed in (0, 1, 2):
        for tr_c, te_c in grouped_folds(dev_vis["client_id"].values, seed):
            tr, te = dev_vis.iloc[tr_c], dev_vis.iloc[te_c]
            Xtr, ncols, _ = features(tr)
            Xte, _, _ = features(te)
            cols = ncols
            if extra:
                Xtr = pd.concat([Xtr, tr[extra]], axis=1)
                Xte = pd.concat([Xte, te[extra]], axis=1)
                cols = ncols + extra
            pre = ColumnTransformer([
                ("n", Pipeline([("i", SimpleImputer(strategy="constant", fill_value=0)),
                                ("s", StandardScaler())]), cols),
                ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats)])
            m = Pipeline([("pre", pre), ("lr", LogisticRegression(class_weight="balanced",
                          max_iter=2000, C=1.0, random_state=42))])
            m.fit(Xtr, tr["is_declining"].values)
            s = m.predict_proba(Xte)[:, 1]
            a.append(roc_auc_score(te["is_declining"].values, s))
            p.append(precision_at_k(s, te["is_declining"].values))
    return np.mean(a), np.mean(p)

legit = auc_with([])
leak = auc_with(leaked)
print(pd.DataFrame([
    ["leakage-safe features (what I submit)", round(legit[0], 3), round(legit[1], 3)],
    ["plus banned label-window columns", round(leak[0], 3), round(leak[1], 3)],
], columns=["feature set", "ROC_AUC", "P@50"]).to_string(index=False))

                          feature set  ROC_AUC  P@50
leakage-safe features (what I submit)    0.602 0.791
     plus banned label-window columns    0.931 1.000


That 0.90 is not a better model, it is a model reading columns computed over the same window the label comes from. At the moment a real refresh decision is made those columns do not exist yet, so a model that leans on them scores well in a notebook and fails in production. The split design and the excluded-column list exist precisely to keep that number honest.

So I report the widened Logistic Regression as a decision-support ranker: it helps most where the workflow reads, Precision@50 on unseen clients, and it does not claim to explain why a page declines. The full search, every method tried and the keep-or-revert decision for each, is recorded under `work/experiments/` alongside the harness that produced it.

## Self-check

- [x] Every section is filled: the reasoning in markdown and the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] Claims stay careful: observed, measured, directional, decision-support
- [x] The split keeps whole clients together, with a lockbox read once at the end